# NeqSim CO2 Impurity Kinetics: CSTR Dynamic Experiment Tutorial

This interactive notebook demonstrates how to set up, configure, and execute dynamic **Continuous Stirred-Tank Reactor (CSTR)** experiments for impurity reactions in dense liquid $\text{CO}_2$ streams.

### Covered Capabilities:
1. **Model Initialization & Custom Setters**:
   - `model.set_reactor_geometry(diameter_cm, volume_ml, mass_flow_g_h)`
   - `model.set_reaction_constants(reaction_id, A_forward, Ea_forward_kJ_mol)`
2. **Getter Methods & Automated Reporting**:
   - `model.generate_reactor_report()` (Reactor geometry & residence time derivation)
   - `model.get_fluid_properties()` (SRK EOS fluid density, phase, fugacities)
   - `model.get_reaction_rates()` ($k_f$ and $K_{\text{eq}}$)
   - `model.get_reactor_geometry()` (Volume, area, length $L$, residence time $\tau$)
3. **Master 50-Hour CSTR Experiment**:
   - **Phase 0 (0 to 10 h)**: Pressurization from $1\text{ bar}, 25^\circ\text{C }\text{N}_2$ to $25\text{ bar}, -25^\circ\text{C}$ liquid $\text{CO}_2$, with outflow opened at $t = 6.35\text{ h}$ to maintain constant pressure & volume.
   - **Phase 1 (10 to 30 h)**: Injection of $10\text{ ppm}$ feed **WITHOUT $\text{H}_2\text{S}$** ($0\text{ ppm }\text{H}_2\text{S}$) for 20 hours.
   - **Phase 2 (30 to 50 h)**: Injection of **ALL $10\text{ ppm}$ impurities INCLUDING $\text{H}_2\text{S}$** for 20 hours.
4. **Master Results Table**:
   - `model.get_table_results(sim_results, resolution_hours=2.0)` (Formatted table with 2-hour resolution).

In [ ]:
import numpy as np
import pandas as pd
from neqsim_co2_kinetics import CO2ImpurityKineticsModel

print("NeqSim CO2 Kinetic Engine Loaded Successfully!")

## 1. Model Setup & Custom Parameter Setters

We initialize the model at $-25^\circ\text{C}$ ($248.15\text{ K}$) and $25\text{ bar}$ in Carbon Steel material, then customize geometry and kinetic parameters using setters.

In [ ]:
# Initialize model
model = CO2ImpurityKineticsModel(T_kelvin=248.15, P_bar=25.0, water_ppm=10.0, material='carbon_steel')

# Set custom reactor geometry: Diameter = 6.5 cm, Volume = 300 mL, Flow Rate = 50 g/h
model.set_reactor_geometry(diameter_cm=6.50, volume_ml=300.0, mass_flow_g_h=50.0)

# Set custom kinetic rate constants (A and Ea)
model.set_reaction_constants("SO2 + H2S + NO2 + O2 -> H2SO4", A_forward=2.13e8, Ea_forward_kJ_mol=15.0)
model.set_reaction_constants("H2S + 3 NO2 <-> SO2 + H2O + 3 NO", A_forward=5.0e7, Ea_forward_kJ_mol=28.0)

print("Custom Setters Applied Successfully!")

## 2. Getter Methods & Automated Reporting

You can retrieve fluid properties, reaction kinetics, and reactor geometry using dedicated getter methods, or generate the complete automated reactor report.

In [ ]:
# 1. Automated Printable Reactor Report
report = model.generate_reactor_report()
print("=" * 100)
print(report)
print("=" * 100)

# 2. Get Fluid Properties Getter
fluid_props = model.get_fluid_properties()
print("\nFluid Properties Getter:")
print(f"  • Fluid Phase:        {fluid_props['phase'].upper()}")
print(f"  • Fluid Mass Density: {fluid_props['mass_density_kg_m3']:.2f} kg/m3 ({fluid_props['molar_density_kmol_m3']:.4f} kmol/m3)")

# 3. Get Reactor Geometry Getter
geom = model.get_reactor_geometry()
print("\nReactor Geometry Getter:")
print(f"  • Calculated Length (L): {geom['length_cm']:.4f} cm ({geom['length_m']:.6f} m)")
print(f"  • Residence Time (tau):  {geom['residence_time_hours']:.4f} HOURS ({geom['residence_time_seconds']:.1f} s)")

## 3. Running the Master 50-Hour Experiment Protocol

We simulate the 3 experimental phases sequentially:
- **Phase 0 (0 to 10 h)**: Pressurization starting with $1\text{ bar}, 25^\circ\text{C }\text{N}_2$ in $300\text{ mL}$ autoclave. Liquid target reached at $t = 6.35\text{ h}$, opening CSTR outflow ($50\text{ g/h}$) to keep $25\text{ bar}, -25^\circ\text{C}$ constant up to $t = 10\text{ h}$.
- **Phase 1 (10 to 30 h)**: 20-hour CSTR injection of $10\text{ ppm}$ feed **WITHOUT $\text{H}_2\text{S}$** ($0\text{ ppm }\text{H}_2\text{S}$).
- **Phase 2 (30 to 50 h)**: 20-hour CSTR injection of **ALL $10\text{ ppm}$ impurities INCLUDING $\text{H}_2\text{S}$**.

In [ ]:
geom = model.get_reactor_geometry()
tau_sec = geom['residence_time_seconds']

# Phase 0: Pressurization & Constant Volume Overflow (0 to 10 h)
pure_co2 = {s: 0.0 for s in model.SPECIES}
res0_fill = model.simulate(initial_ppm=pure_co2, duration_sec=6.3463*3600.0, num_points=64, feed_ppm=pure_co2, space_time_sec=None)

state_6h = {s: res0_fill['ppm'][s][-1] for s in model.SPECIES}
res0_flow = model.simulate(initial_ppm=state_6h, duration_sec=(10.0 - 6.3463)*3600.0, num_points=37, feed_ppm=pure_co2, space_time_sec=tau_sec)

# Phase 1: Injection Without H2S for 20 Hours (10 to 30 h)
state_10h = {s: res0_flow['ppm'][s][-1] for s in model.SPECIES}
feed_p1 = {'H2S': 0.0, 'SO2': 10.0, 'NO2': 10.0, 'O2': 10.0, 'H2O': 10.0}
res1 = model.simulate(initial_ppm=state_10h, duration_sec=20.0*3600.0, num_points=201, feed_ppm=feed_p1, space_time_sec=tau_sec)

# Phase 2: Injection With All Impurities for 20 Hours (30 to 50 h)
state_30h = {s: res1['ppm'][s][-1] for s in model.SPECIES}
feed_p2 = {'H2S': 10.0, 'SO2': 10.0, 'NO2': 10.0, 'O2': 10.0, 'H2O': 10.0}
res2 = model.simulate(initial_ppm=state_30h, duration_sec=20.0*3600.0, num_points=201, feed_ppm=feed_p2, space_time_sec=tau_sec)

# Combine timeline
all_t = np.concatenate([res0_fill['time_hours'], 6.3463 + res0_flow['time_hours'], 10.0 + res1['time_hours'], 30.0 + res2['time_hours']])
all_ppm = {s: np.concatenate([res0_fill['ppm'][s], res0_flow['ppm'][s], res1['ppm'][s], res2['ppm'][s]]) for s in model.SPECIES}

master_sim = {'time_hours': all_t, 'ppm': all_ppm}
print("50-Hour Master Experiment Simulation Completed Successfully!")

## 4. Master Concentration Table (2-Hour Resolution)

We call `model.get_table_results(master_sim, resolution_hours=2.0)` to generate and format the master output table at 2-hour intervals.

In [ ]:
# Generate 2-hour resolution table
df_master = model.get_table_results(master_sim, resolution_hours=2.0)

print("=" * 110)
print("MASTER 50-HOUR EXPERIMENT CONCENTRATION TABLE (2-HOUR RESOLUTION)")
print("=" * 110)
df_master